# Matrix Multiplication: Numpy vs RV32IMP DUT
Dự án: RV32IM P-Extension DSP Implementation

Notebook này thực hiện:
1. **Sinh dữ liệu**: Tạo ma trận A (32x32) và B (32x32) kiểu int8.
2. **Golden Model**: Tính toán ma trận C = A x B bằng Numpy (int32).
3. **Xuất dữ liệu**: 
   - `matrix_data.h`: Header file chứa mảng C cho mã nguồn C.
   - `golden_w.hex`: File hex chứa kết quả mong đợi để so sánh trong Testbench RTL.

In [23]:
import numpy as np
import os

# Cấu hình
N = 32
np.random.seed(42)

# 1. Sinh ma trận ngẫu nhiên int8
mat_A = np.random.randint(-128, 127, (N, N), dtype=np.int8)
mat_B = np.random.randint(-128, 127, (N, N), dtype=np.int8)

# 2. Tính toán Golden Result (int32)
mat_C = np.matmul(mat_A.astype(np.int32), mat_B.astype(np.int32))

print(f"Ma trận A: {mat_A.shape}, B: {mat_B.shape}, C: {mat_C.shape}")
print("Ví dụ kết quả C[0,0]:", mat_C[0,0])

Ma trận A: (32, 32), B: (32, 32), C: (32, 32)
Ví dụ kết quả C[0,0]: 42912


In [25]:
def save_hex(filename, data, words_per_line=4):
    flatten_data = data.flatten()
    with open(filename, 'w') as f:
        for i, val in enumerate(flatten_data):
            # Chuyển sang hex 32-bit unsigned
            hex_val = f"{(int(val) & 0xFFFFFFFF):08x}"
            f.write(hex_val)
            if (i + 1) % words_per_line == 0:
                f.write('\n')
            else:
                f.write('\t')
        if len(flatten_data) % words_per_line != 0:
            f.write('\n')

# 3. Lưu file HEX cho Simulation
os.makedirs('data', exist_ok=True)
save_hex('golden_w.hex', mat_C)
print("Đã lưu golden_w.hex")

Đã lưu golden_w.hex


In [26]:
# 4. Sinh file matrix_data.h cho C code
with open('matrix_data.h', 'w') as f:
    f.write("#ifndef MATRIX_DATA_H\n")
    f.write("#define MATRIX_DATA_H\n\n")
    f.write("#include <stdint.h>\n\n")
    f.write(f"#define MAT_N {N}\n")
    f.write(f"#define H_A   MAT_N\n")
    f.write(f"#define W_A   MAT_N\n")
    f.write(f"#define W_B   MAT_N\n\n")
    
    f.write("static const int8_t mat_A[H_A * W_A] __attribute__((aligned(4))) = {\n")
    a_flat = mat_A.flatten()
    for i, val in enumerate(a_flat):
        f.write(f"{val:4}")
        if i < len(a_flat) - 1: f.write(",")
        if (i + 1) % 16 == 0: f.write("\n")
    f.write("};\n\n")
    
    f.write("static const int8_t mat_B[W_A * W_B] __attribute__((aligned(4))) = {\n")
    b_flat = mat_B.flatten()
    for i, val in enumerate(b_flat):
        f.write(f"{val:4}")
        if i < len(b_flat) - 1: f.write(",")
        if (i + 1) % 16 == 0: f.write("\n")
    f.write("};\n\n")
    f.write("#endif // MATRIX_DATA_H\n")

print("Đã lưu matrix_data.h")

Đã lưu matrix_data.h
